In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# ============================================================
# 1. KPI: MTBF
# ============================================================
fact_faults = spark.read.table(f"{GOLD}.fact_fault_event")

# fault_ts string (ISO-8601) hai — pehle timestamp me convert karo
fact_faults = fact_faults.withColumn("fault_ts", try_to_timestamp("fault_ts"))

w_faults = Window.partitionBy("machine_sk").orderBy("fault_ts")
mtbf_df = fact_faults.select("machine_sk", "fault_ts") \
    .withColumn("prev_fault", lag("fault_ts").over(w_faults)) \
    .withColumn("hours_diff", (unix_timestamp("fault_ts") - unix_timestamp("prev_fault")) / 3600.0) \
    .filter(col("prev_fault").isNotNull()) \
    .groupBy("machine_sk").agg(avg("hours_diff").alias("mtbf_hours"))

mtbf_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.kpi_mtbf")

# ============================================================
# 2. KPI: MTTR
# ============================================================
work_orders = spark.read.table(f"{SILVER}.work_order")

work_orders = work_orders.withColumn("start_time", try_to_timestamp("start_ts")) \
                         .withColumn("end_time", try_to_timestamp("end_ts"))

work_orders = work_orders.withColumn("repair_hours", 
    when(col("start_time").isNull() | col("end_time").isNull(), None)
    .otherwise((unix_timestamp("end_time") - unix_timestamp("start_time")) / 3600.0)
)

# Negative repair_hours ko 0 kar do
work_orders = work_orders.withColumn("repair_hours", when(col("repair_hours") < 0, 0).otherwise(col("repair_hours")))

mttr_df = work_orders.groupBy("machine_id").agg(avg("repair_hours").alias("mttr_hours")) \
    .join(spark.read.table(f"{GOLD}.dim_machine").select("machine_sk", "machine_id"), "machine_id", "left") \
    .select("machine_sk", "mttr_hours")

mttr_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.kpi_mttr")

print("✅ gold_06_kpi_reliability built successfully!")